In [1]:
import os
import json
import random
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import time


# Load dataset

In [2]:
# Set Paths
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

def load_dataset(qa_json_path, description_csv_path):
    try:
        # Load QA data
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]
        
        # Load descriptions
        descriptions = pd.read_csv(description_csv_path)
        
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()

def get_seeded_question(questions, gif_num, base_seed=42):
    """
    Get a deterministic random question for a GIF using fixed seed
    
    Args:
        questions: List of questions for current GIF
        gif_num: Current GIF number
        base_seed: Base random seed
        
    Returns:
        Selected question entry
    """
    if not questions:
        return None
    # Create new Random instance for each GIF
    local_random = random.Random(base_seed + gif_num)
    # Sort questions to ensure consistent ordering
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

# Load data
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# TODO 修改，增加别的video
# Filter questions for episode 1
ep1_questions = [q for q in qa_data if "Pororo_ENGLISH1_1_ep1" in q["video_name"]]


# Single agent prediction

In [3]:
load_dotenv()

# Configuration
# MODEL_NAME = "gpt-4o-mini" 
MODEL_NAME = "claude-3-5-haiku-20241022" 

# Determine which platform to use based on the model name
is_openai_model = not MODEL_NAME.startswith("claude-")

# Initialize appropriate client
if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

def get_prediction(question, gif_paths, description, subtitles, max_retries=3, retry_delay=2):
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())

    prompt = f"""
Please answer the following question based on the given context:

Question: {question}
Scene Description: {description}
Subtitles: {subtitles}

Guidelines for answering:
1. Focus on answering the specific question asked
2. Include all relevant details from the context
3. Maintain accuracy while being clear
4. Avoid unnecessary elaboration
5. Keep the answer complete but concise

Your answer should be factual and directly address the question.
"""
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                # OpenAI implementation
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.3,
                )
                return completion.choices[0].message.content.strip()
            else:
                # Anthropic implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.3,
                )
                return completion.content[0].text.strip()

        except Exception as e:
            print(f"Prediction attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print(f"All {max_retries} attempts failed for question: {question}")
    return None

Using Anthropic model: claude-3-5-haiku-20241022


# Compute accuracy

In [4]:
def compute_accuracy(correct_answer, predicted_answer, max_retries=2, retry_delay=2):
    """
    Compare the correct answer with the predicted answer using semantic similarity
    Returns: float between 0 and 1 indicating the similarity/correctness
    """
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                # OpenAI implementation
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=10,
                    temperature=0.3
                )
                score = float(completion.choices[0].message.content.strip())
            else:
                # Anthropic implementation
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=10,
                    temperature=0.3
                )
                score = float(completion.content[0].text.strip())

            # Ensure score is between 0 and 1
            return max(0.0, min(1.0, score))

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    # Return 0 if we can't parse the score
    return 0.0


# Evaluate model performance

In [5]:
# Create list to store evaluation results
evaluation_results = []

# Group questions by supporting_num
grouped_questions = {}
for entry in ep1_questions:
    supporting_num = entry["supporting_num"]
    if supporting_num not in grouped_questions:
        grouped_questions[supporting_num] = []
    grouped_questions[supporting_num].append(entry)

# Process GIFs 1-36 in sequence
gif_numbers = list(range(1, 37))
correct_count = 0
total_count = len(gif_numbers)
gif_accuracy = {}

# Process each GIF number in sequence
for gif_num in tqdm(gif_numbers, total=total_count):
    current_gif_questions = grouped_questions.get(str(gif_num), [])
    
    if not current_gif_questions:
        print(f"No questions found for GIF {gif_num}")
        continue
        
    # Use seeded random selection
    entry = get_seeded_question(current_gif_questions, gif_num)
    
    # Get question info
    video_name = entry["video_name"]
    question = entry["question"]
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Construct paths
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", "Pororo_ENGLISH1_1", "Pororo_ENGLISH1_1_ep1")
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    # Load subtitles
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    # Process current GIF
    gif_files = [f"{gif_num}.gif"]
    gif_paths = [os.path.join(episode_folder, gif_file) for gif_file in gif_files]

    # Get description
    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    # Get prediction
    predicted_answer = get_prediction(question, gif_paths, description, subtitles)

    # Calculate accuracy
    is_correct = predicted_answer is not None and compute_accuracy(correct_answer, predicted_answer)
    correct_count += is_correct

    # Store current result
    result = {
        'gif_num': gif_num,
        'video_name': video_name,
        'qid': qid,
        'question': question,
        'correct_answer': correct_answer,
        'predicted_answer': predicted_answer,
        'accuracy': (is_correct) 
    }
    evaluation_results.append(result)

    # Print debugging info
    print(f"Video name: {video_name}")
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"GIF number: {gif_num}, Accuracy: {(is_correct):.4f}")

# Calculate overall accuracy
average_accuracy = correct_count / total_count
print(f"Average Accuracy: {average_accuracy:.4f}")

  3%|▎         | 1/36 [00:03<02:18,  3.95s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 1
QID: 4
Question: what is the nature of place where pororo leaves?
Correct Answer: it is the cold country , small village
Predicted Answer: Based on the context provided, Pororo lives in a cold, snowy forest located in a small village on a mountainous region. The scene description specifically mentions trees covered in snow on mountains, and the subtitles confirm it is a "cold country" with a forest setting.
GIF number: 1, Accuracy: 0.7500


  6%|▌         | 2/36 [00:05<01:36,  2.83s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 2
QID: 83
Question: what is the name of the little penguin
Correct Answer: the name of the little penguin is pororo
Predicted Answer: Based on the context provided, the name of the little penguin is Pororo.
GIF number: 2, Accuracy: 1.0000


  6%|▌         | 2/36 [00:07<02:07,  3.76s/it]


KeyboardInterrupt: 

# Save data

In [ ]:
# Save results with explicit file handling to ensure overwriting works
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average Accuracy']

# Ensure no duplicate summary rows when saving
unique_questions = len(set(r['gif_num'] for r in evaluation_results))

# Add average accuracy as the last row
average_result = {
    'gif_num': 'Average Accuracy',
    'video_name': f'Total Questions: {unique_questions}',
    'qid': '',
    'question': '',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
evaluation_results.append(average_result)

# Define column order
column_order = [
    'gif_num',
    'video_name',
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Create a safe filename using the model name
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Set up output directory and path
# results_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/results"
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(results_dir, f'pororo_evaluation_results_single_agent_{safe_model_name}.csv')

# First, check if file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure proper closure
    results_df.to_csv(output_path, index=False)
    
    # Verify file creation
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
        print(f"Total questions: {unique_questions}")
        print(f"Average accuracy: {average_accuracy:.4f}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")